$$
\text{Attention}(Q,K,V) = \text{softmax}\left( \frac{QK^T}{\sqrt{d_k}} \right) V
$$
Q：Query 查询矩阵；K：Key 键矩阵；V：Value 值矩阵；
$d_k$：Q、K向量的特征维度

1. $QK^T$：计算查询向量与全部键向量的点积，得到相似度矩阵。
   单个点积 $q\cdot k=\sum_{i=1}^{d_k} q_i k_i$。
   假设$q,k$各分量独立，均值0、方差1：
   $$\text{Var}(q\cdot k) = \text{Var}\big(\sum q_i k_i\big)=\sum \text{Var}(q_i k_i)=d_k$$
   特征维度$d_k$越大，点积方差越大，数值范围越宽。

2. 除以$\sqrt{d_k}$缩放推导：
   $$s=\frac{q\cdot k}{\sqrt{d_k}},\quad \text{Var}(s)=\frac{\text{Var}(q\cdot k)}{d_k}=\frac{d_k}{d_k}=1$$
   缩放后相似度方差固定为1，不受$d_k$大小影响。
   不缩放：$d_k$很大时点积数值剧烈增大，softmax进入饱和区，梯度几乎消失。

3. softmax：将原始相似度转换为总和等于1的注意力权重
4. 权重矩阵 × V：使用注意力权重对Value做加权求和，得到注意力输出


**（交叉注意力）场景**：从 5 部候选电影里，选出甲最想看的一部。目前已有甲的用户向量和已看过的 3 部电影的信息。每一部电影都有各自的信息，比如类型、评分等。

**分析**：
- 第一步：甲用自身的偏好向量，分别比对 3 部已看电影的类型特征，算出每一部与甲的匹配度。比如三部匹配度分别为 1、3、2。
- 第二步：由于是针对同一个甲的匹配度，这里将其做归一化，让所有权重总和为 1，得到注意力分配比例，对应为 1/6、3/6、2/6。
- 第三步：根据这个比例，分别抽取每部电影对应的完整信息向量，再全部相加，融合得到一个结合了甲观影历史的新向量。
- 第四步：用这个新向量去匹配 5 部候选电影，选出最推荐的影片；后续还可以叠加残差连接等网络处理。

**总结**：
- 甲 = Qurey；电影 = Key、Value
- **用 Query 去和所有 Key 做相似度匹配，再用匹配得到的权重，把对应的 Value 加权融合起来**。Key 管 “对不对得上”，Value 管 “拿什么内容来融合”，二者是同一个输入对象拆出来的两个功能角色。

剩下的缩放、Softmax 归一化这些都属于配套的工程细节：缩放是为了抵消特征维度对数值范围的影响，Softmax 是把原始匹配分转成总和为 1 的概率型权重，都是为了让 “按相似度加权” 这个过程更稳定、更符合神经网络的训练需求。

按 “目标→核心组件→增量细节” 的思路去理解就很清晰，不用纠结黑盒里的精细调优，抓住 “相似度匹配 + 加权聚合” 这个主干就够了。

==================================================================================================================================

（**自注意力）场景**
短句：苹果很甜。每个词单独看都有多种含义，通过自注意力结合整句上下文，确定每个词在句中的准确含义。

**分析**
1. **处理第一个词「苹果」**
   - 第一步：以「苹果」为基准，分别比对句中3个词的语义关联程度，得到匹配度：3、1、2
   - 第二步：将这组匹配度做归一化，让所有权重总和为 1，得到注意力分配比例：3/6、1/6、2/6
   - 第三步：按照这个比例融合每个词的语义信息，得到「苹果」的句中含义——特指水果苹果，排除手机品牌等其他歧义

2. **处理第二个词「很」**
   - 第一步：以「很」为基准，分别比对句中3个词的语义关联程度，得到匹配度：1、3、2
   - 第二步：将这组匹配度做归一化，得到注意力分配比例：1/6、3/6、2/6
   - 第三步：按照这个比例融合语义信息，得到「很」的句中含义——用来修饰甜度的高低程度

3. **处理第三个词「甜」**
   - 第一步：以「甜」为基准，分别比对句中3个词的语义关联程度，得到匹配度：2、1、3
   - 第二步：将这组匹配度做归一化，得到注意力分配比例：2/6、1/6、3/6
   - 第三步：按照这个比例融合语义信息，得到「甜」的句中含义——形容水果的口感味道

4. **最终输出与下游应用**
   三个词全部处理完成后，输出3个与输入数量一致、且各自携带整句上下文信息的语义结果，可直接用于下游任务：
   - 机器翻译场景：准确译为 `The apple is very sweet`，不会错误翻译成电子产品相关内容
   - 语义检索场景：会被匹配到「水果口感」类目，不会关联到数码产品

   后续也可叠加残差连接、层归一化等网络结构进一步优化特征。

**总结**
- 句中每个词轮流充当 **Query**，同时也都作为 **Key、Value**
- 核心逻辑：每个元素轮流做一次查询基准，和所有元素做相似度匹配，再用匹配得到的权重加权融合对应内容。输入多少个元素，输出就有多少个元素，每个结果都融入了全局上下文信息，可支撑各类语义理解任务。
